### **Algorithme d'Apprentissages : Projet Foot - Vladimir Varenne-Welnovski**
Ce notebook est le rendu final pour le projet d'Algorithme d'Apprentissage : son but est d'explorer les solutions pour predire le resultat d'un match de foot.

Il se decompose en une partie d'exploration qualitative des donnees, puis en une partie prediction visant a tester differentes methodes pour arriver a predire avec le moins d'erreur possible le resultat des matchs de Ligue 1.(rajouter un montant d'argent si on avait bet sur chq game ?)

In [1]:
# Importation des library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Style global des graphs

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f8f8',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'sans-serif',
    'axes.titlesize': 14,
    'axes.labelsize': 11,
})

#### **PARTIE 1 : EXPLORATION**
Voici les .csv que nous avons a notre disposition : 
- match_2013_2024.csv : La liste de tous les matchs avant le début de saison 2025-2026, avec le résultat associé
- match_2025.csv : La liste des matchs à prédire pour la saison 2025-2026, vous n’avez pas le résultat, il sera dévoilé après la fin du projet.
- sample_results.csv : un exemple de fichier de prédiction que j’attends pour tester votre modèle. À chaque id de match est associé un résultat (-1 0 ou 1)

- clubs_fr.csv : Une liste des clubs français avec quelques stats sur la constitution (récente) de l’équipe
- player_valuation_before_season.csv : Pour chaque joueur, sa valeur sur le marché, à une certaine date, à vous de transformer et d’utiliser cette information à bonne escient.
- player_appearance.csv : Un ensemble d’info simple sur chaque joueur pendant chaque match (nombre de buts, nombre de cartons)
- game_lineups.csv : La constitution des équipes pour chaque match, vous n’avez pas la constitution pour les matchs récents.
- game_events.csv :  Un ensemble d’actions pour chaque joueur pendant chaque match, peut compléter les informations sur un joueur, vous allez sûrement devoir aggreger des informations ce jeu (par exemple comptez le nombre de passe décisive, d’arrêt etc...)

In [12]:
matches = pd.read_csv("data/matchs_2013_2024.csv", index_col=0)
#df['results'].value_counts()
matches.head()

,game_id,season,round,date,home_club_id,away_club_id,home_club_goals,away_club_goals,home_club_position,away_club_position,...,stadium,attendance,referee,home_club_formation,away_club_formation,home_club_name,away_club_name,aggregate,competition_type,results
523,2223841,2012,6. Matchday,2012-09-22,3911,1423,2,1,8.0,9.0,...,Stade Francis-Le Blé,10627.0,Sébastien Moreira,NaN,NaN,Stade brestois 29,Valenciennes FC,2:1,domestic_league,1
524,2223842,2012,7. Matchday,2012-09-29,1147,3911,1,0,9.0,10.0,...,Stade François-Coty,6029.0,Olivier Thual,NaN,NaN,AC Ajaccio,Stade brestois 29,1:0,domestic_league,1
525,2223843,2012,6. Matchday,2012-09-22,1421,1159,2,0,5.0,19.0,...,Stade Auguste-Delaune,13413.0,Jean-Charles Cailleux,NaN,NaN,Stade Reims,AS Nancy-Lorraine,2:0,domestic_league,1
526,2223844,2012,6. Matchday,2012-09-21,969,618,1,1,16.0,10.0,...,Stade de la Mosson,16666.0,Lionel Jaffredo,NaN,NaN,Montpellier HSC,AS Saint-Étienne,1:1,domestic_league,0
527,2223845,2012,6. Matchday,2012-09-23,1082,1041,1,1,12.0,2.0,...,Decathlon Arena-Stade Pierre-Mauroy,43475.0,Clément Turpin,NaN,NaN,Lille Olympique Sporting Club,Olympique Lyonnais,1:1,domestic_league,0


In [13]:
matches.shape

(4691, 22)

#### Investigating Missing Data
Historiquement, en Ligue 1, il y a 20 equipes dans le championnat. Vu qu'elles affronte chaque autre equipe 1 fois a domicile (et que le csv ne prend en compte que les matchs de "domestic_league", et pas les matchs de Coupe de France etc.), le nombre de matchs en 1 saison devrait systematiquement de 20*19. Or, une reforme de la LFP a fait passer le nombre de clubs present en Ligue 1 a 18 (et un passage de 38 a 36 journees) a partir la saison 2023-2024. Cette saison etant presente dans nos donnees, on doit le prendre en compte.
Nous devrions alors avoir : 
2012-2013 20 equipes
2013 2014 20 equipes
2014 2015 20 equipes
2015 2016 20 equipes
2016 2017 20 equipes
2017 2018 20 equipes
2018 2019 20 equipes
2019-2020 20 equipes
2020 2021 20 equipes
2021 2022 20 equipes
2022 2023 20 equipes
2023 2024 18 equipes
2024 2025 18 equipes
20x19 x 11 + 18x17 x 2 matchs

In [39]:
20*19 * 11 + 18*17 * 2

4792

Ce nombre de matchs n'est pas coherent avec le nombre de lignes donne par matches.shape (4691). Y a t-il des matchs manquants dans ce dataset ? 

La colonne "season" me trouble. Je ne sais pas si la saison "2012" correspond a la saison 2011-2012, 2012-2013, ou si une erreur aurait pu s'y mettre en comptant des matchs des deux saisons differentes. Verifions cela : 

In [30]:
saison_2012 = matches[matches["season"] == 2012]
clubs_2012 = pd.concat([
    saison_2012["home_club_name"],
    saison_2012["away_club_name"]
]).unique().tolist()

print(clubs_2012)

['Stade brestois 29', 'AC Ajaccio', 'Stade Reims', 'Montpellier HSC', 'Lille Olympique Sporting Club', 'Toulouse Football Club', 'SC Bastia', 'Football Club Lorient-Bretagne Sud', 'Olympique de Marseille', 'FC Sochaux-Montbéliard', 'FC Girondins Bordeaux', 'Valenciennes FC', 'Paris Saint-Germain Football Club', 'Olympique Lyonnais', 'Stade Rennais Football Club', 'AS Saint-Étienne', "Olympique Gymnaste Club Nice Côte d'Azur", 'AS Nancy-Lorraine', 'ESTAC Troyes', 'Thonon Évian Grand Genève FC']


Cette liste correspond bien aux clubs de Ligue 1 dans la saison 2012-2013 (https://fr.wikipedia.org/wiki/Championnat_de_France_de_football_2012-2013#Classement)
Mais l'ordre dans la colonne "round" n'etant pas ordonnee, je ne suis pas certain qu'il n'y a pas d'erreur dans cette saison. Je repete ensuite cette operation pour toutes les annees.

In [35]:
len(saison_2012)

380

In [34]:
matches.groupby("season").size()

season
2012    380
2013    380
2014    380
2015    380
2016    380
2017    380
2018    380
2019    279
2020    380
2021    380
2022    380
2023    306
2024    306
dtype: int64

Il y a effectivement eu uniquement 279 matchs joues (sur les 380 prevus) en raison de la fin prematuree du championnat causee par la pandemie du Covid-19 en France. 

In [ ]:
print(matches["home_club_name"].value_counts())

home_club_name
Lille Olympique Sporting Club                   239
Olympique Gymnaste Club Nice Côte d'Azur        239
Montpellier HSC                                 238
Olympique de Marseille                          238
Paris Saint-Germain Football Club               238
Stade Rennais Football Club                     238
Olympique Lyonnais                              237
Football Club de Nantes                         219
Association sportive de Monaco Football Club    219
AS Saint-Étienne                                202
Stade Reims                                     200
Toulouse Football Club                          199
FC Girondins Bordeaux                           184
Football Club Lorient-Bretagne Sud              169
Angers Sporting Club de l'Ouest                 165
Racing Club de Strasbourg Alsace                142
Football Club de Metz                           126
Stade brestois 29                               124
EA Guingamp                                     1

In [ ]:
saison_2019 = matches[matches["season"] == 2019]
print(saison_2019["home_club_name"].value_counts())

home_club_name
Angers Sporting Club de l'Ouest                 15
Lille Olympique Sporting Club                   15
Olympique Gymnaste Club Nice Côte d'Azur        15
Association sportive de Monaco Football Club    14
Dijon FCO                                       14
Montpellier HSC                                 14
Olympique de Marseille                          14
Paris Saint-Germain Football Club               14
Stade brestois 29                               14
Football Club de Metz                           14
AS Saint-Étienne                                14
Nîmes Olympique                                 14
Football Club de Nantes                         14
Amiens SC                                       14
Stade Reims                                     14
Stade Rennais Football Club                     14
Racing Club de Strasbourg Alsace                13
Toulouse Football Club                          13
Olympique Lyonnais                              13
FC Girondins Bor